# One fixed offset diversity blend v012

Exactly one candidate was declared before calculation in `research/ENSEMBLE_HYPOTHESIS_v012.md`: deep1/2 plus original compact, curve, memory and offset each1/8. There is no roster or weight search, and this screen does not select a D candidate.

Default execution reviews an existing report only. Explicit `PA3_RUN_DIVERSITY_V012=1` computes native A metrics from saved predictions and approved labels, verifies the completed v009 archive/source/model/record/prediction hashes and prior58-check v010 audit, and saves results and the one blended prediction array. Existing source notebooks may have different execution outputs; only the exact notebook paths in v010's source manifest can use matching executable-code hashes instead of matching raw bytes. Every exception records expected and observed raw/code evidence; all other file hashes stay strict.

No model/checkpoint is loaded, no feature or model is fitted, no inference/GPU work occurs, and no D split or outcome is accessed or constructed. Targets are parsed as float32 and converted to float64 only for native point-weighted inspiratory metrics. The predeclared .003 gain/negative whole-breath95% bootstrap gate is unchanged. D selection remains pending the completed five fixed comparisons; no promotion, final fit or submission occurs here.


In [1]:
import hashlib,json,os,platform
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
import pandas as pd
V012_RUN=os.environ.get('PA3_RUN_DIVERSITY_V012','0')=='1'
V012_ROOT=Path.cwd().resolve()
V012_NOTEBOOK='research/diversity_screen_v012.ipynb'
V012_OUT='outputs/diversity_screen_v012'
V012_NAME='deep_half_compact_curve_memory_offset'
V012_WEIGHTS=dict(original_deep=.5,original_compact=.125,curve_consistency=.125,learned_memory=.125,shared_breath_offset=.125)
V012_HYPOTHESIS_SHA='973f48afe6807c4277e9d498f44927371c4dddb88748045a485a58ece0d2b595'
V012_PRIOR_RESULTS_SHA='949999795892ada6a2b28191df998ef3548a4772ae940e2450dee02667eea7bd'
V012_PRIOR_REVIEW_SHA='cc008853f58f299993b78c805d92b98513c92edfa243077e1196692a7e3dd332'
V012_COMPLETED_MANIFEST_SHA='c6b17f6691cb748efe4244369c7122b1f046f8c74ead14289ef6bf4e0bc3ef27'

def v012_file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def v012_array_sha(values):
    return hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()

def v012_code_sha(path):
    notebook = json.loads(Path(path).read_text())
    source = '\n'.join(''.join(cell.get('source', [])) for cell in notebook['cells']
        if cell.get('cell_type') == 'code')
    return hashlib.sha256(source.encode()).hexdigest()

def v012_json_sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def v012_local(root, saved_path):
    value = str(saved_path)
    marker = '/'+root.name+'/'
    path = (root/value.split(marker, 1)[1] if marker in value else root/value).resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise ValueError('Missing artifact or path outside repository: '+value)
    return path

def v012_metrics(prediction, truth, counts):
    error = np.abs(prediction-truth)
    starts = np.r_[0, np.cumsum(counts)[:-1]]
    return dict(mae=float(error.mean()), median_absolute_error=float(np.median(error)),
        p90_absolute_error=float(np.quantile(error, .9)),
        median_breath_mae=float(np.median(np.add.reduceat(error, starts)/counts)))

def v012_paired_interval(new_error, reference_error, counts):
    """Paired whole-breath resampling, retaining point-weighted native MAE."""
    sums = np.add.reduceat(new_error-reference_error, np.r_[0, np.cumsum(counts)[:-1]])
    rng = np.random.default_rng(20261201)
    draws = []
    for _ in range(10):
        selected = rng.integers(0, len(counts), size=(100, len(counts)))
        draws.extend((sums[selected].sum(axis=1)/counts[selected].sum(axis=1)).tolist())
    low, high = np.quantile(draws, [.025, .975])
    return dict(difference=float((new_error-reference_error).mean()),
        bootstrap_95_low=float(low), bootstrap_95_high=float(high), seed=20261201,
        repeats=1000, unit='whole breath', point_count_weighted=True,
        direction='candidate minus original fixed pair; negative favors candidate')

def v012_run_fixed_screen(root=V012_ROOT):
    if os.environ.get('PA3_RUN_DIVERSITY_V012','0')!='1':
        raise RuntimeError('Explicit PA3_RUN_DIVERSITY_V012=1 required')
    root=Path(root).resolve(); directory=root/V012_OUT; destination=directory/'results.json'
    if destination.exists(): raise FileExistsError('Completed V012 screen is preserved')
    before=v012_code_sha(root/V012_NOTEBOOK)
    checks=[]; inputs={}; notebook_output_variations=[]
    def check(name,condition):
        if not bool(condition): raise AssertionError(name)
        checks.append(name)
    def read(path): return json.loads(Path(path).read_text())
    def verify(path,expected=None):
        path=Path(path); observed=v012_file_sha(path); relative=str(path.relative_to(root))
        if expected is not None: check('file_hash:'+relative,observed==expected)
        inputs[relative]=dict(file_sha256=observed)
        return observed
    verify(root/'research/ENSEMBLE_HYPOTHESIS_v012.md',V012_HYPOTHESIS_SHA)
    prior_path=root/'outputs/diversity_screen_v010/results.json'
    verify(prior_path,V012_PRIOR_RESULTS_SHA); prior=read(prior_path)
    review_path=root/'research/diversity_review_v010.json'
    verify(review_path,V012_PRIOR_REVIEW_SHA); review=read(review_path)
    check('prior_independent58checks',review['status']=='passed' and review['check_count']==58
        and review['results_sha256']==V012_PRIOR_RESULTS_SHA and all(review['checks'].values()))
    check('prior_code_hash',v012_code_sha(root/'research/diversity_screen_v010.ipynb')
        ==prior['code_hash_before_run']==prior['code_hash_after_run']==review['source_code_sha256'])
    for relative,item in prior['verified_inputs'].items():
        path=v012_local(root,relative); observed=v012_file_sha(path)
        if observed!=item['file_sha256']:
            manifest=prior['source_manifest'].get(relative)
            check('only_whitelisted_notebook_output_variation:'+relative,
                manifest is not None and relative.endswith('.ipynb')
                and v012_code_sha(path)==manifest['code_sha256'])
            notebook_output_variations.append(dict(path=relative,expected_raw_sha256=item['file_sha256'],
                observed_raw_sha256=observed,expected_code_sha256=manifest['code_sha256'],
                observed_code_sha256=v012_code_sha(path)))
        else: check('prior_strict_file_hash:'+relative,True)
        inputs[relative]=dict(file_sha256=observed)
    base=root/'outputs/improvements_v005/sequence'; out9=root/'outputs/controlled_experiments_v009'
    manifest_path=out9/'completed-backup-manifest.json'
    verify(manifest_path,V012_COMPLETED_MANIFEST_SHA); completed=read(manifest_path)
    archived=out9/'server_snapshot_completed'; archive_index={item['path']:item for item in completed['files']}
    check('complete_archive_manifest46entries',len(completed['files'])==len(archive_index)==46)
    for relative,item in archive_index.items():
        path=v012_local(root,archived/relative)
        verify(path,item['sha256']); check('archive_file_size:'+relative,path.stat().st_size==item['bytes'])
    for relative in ['outputs/controlled_experiments_v009/design.json',
            'outputs/controlled_experiments_v009/screen_summary.json',
            'outputs/controlled_experiments_v009/A_row_ids.npy',
            'outputs/controlled_experiments_v009/record_A_shared_breath_offset.json',
            'outputs/controlled_experiments_v009/shared_breath_offset/model.pt',
            'outputs/controlled_experiments_v009/shared_breath_offset/report.json',
            'outputs/controlled_experiments_v009/A_shared_breath_offset_predictions.npy']:
        verify(root/relative,archive_index[relative]['sha256'])
    design=read(out9/'design.json'); summary=read(out9/'screen_summary.json')
    check('v009_both_stages_complete_no_D',summary['status']=='complete' and len(summary['results'])==2
        and {item['name'] for item in summary['results']}=={'shared_breath_offset','full_context_tcn'}
        and summary['selected_for_confirmation'] is None and summary['confirmation_not_performed'])
    source_manifest={}
    for filename,expected in design['component_hashes'].items():
        current=root/'research'/filename; snapshot=archived/'research'/filename
        check('v009_component_code_hash:'+filename,
            v012_code_sha(current)==expected==v012_code_sha(snapshot))
        source_manifest['research/'+filename]=dict(code_sha256=expected,file_sha256=verify(current))
    check('v009_main_code_hash',v012_code_sha(root/'controlled_experiments_v009.ipynb')
        ==design['source_sha256']==v012_code_sha(archived/'controlled_experiments_v009.ipynb'))
    source_manifest['controlled_experiments_v009.ipynb']=dict(code_sha256=design['source_sha256'],
        file_sha256=verify(root/'controlled_experiments_v009.ipynb'))
    verify(root/'research/authorization_v009.json',design['authorization_sha256'])
    check('standing_approved_data_authorization',read(root/'research/authorization_v008.json')['standing_approval_for_controlled_iterations'])
    verify(root/'data/train.csv',design['train_sha256'])
    original=read(base/'design.json')
    check('approved_original_data_identity',design['train_sha256']==original['train_sha256'])
    frame=pd.read_csv(root/'data/train.csv',usecols=['id','breath_id','u_out','pressure'],
        dtype=dict(id='int64',breath_id='int32',u_out='int8',pressure='float32'))
    check('approved_row_count',len(frame)==67905*80)
    blocks=frame.breath_id.to_numpy().reshape(-1,80); breath_ids=blocks[:,0]
    check('unique_contiguous_breaths',np.all(blocks==breath_ids[:,None])
        and len(np.unique(breath_ids))==67905 and frame.id.is_unique)
    with np.load(base/'split_breath_ids.npz',allow_pickle=False) as split:
        validation=np.sort(pd.Index(breath_ids).get_indexer(split['A_validation']))
    check('exact_original_A_group',len(validation)==6791 and (validation>=0).all()
        and v012_array_sha(breath_ids[validation])==original['group_hashes']['A_validation']
        ==design['plan']['group_hashes']['A_validation'])
    valve=frame.u_out.to_numpy().reshape(-1,80)[validation]; mask=valve==0; counts=mask.sum(1)
    check('valid_A_phase_mask',np.isin(valve,[0,1]).all() and (valve[:,0]==0).all()
        and (np.diff(valve,axis=1)>=0).all() and (counts>0).all())
    truth32=frame.pressure.to_numpy().reshape(-1,80)[validation][mask]
    truth=truth32.astype(np.float64); row_ids=frame.id.to_numpy().reshape(-1,80)[validation][mask]
    row_hash=v012_array_sha(row_ids)
    check('exact_A_native_truth',truth32.dtype==np.float32 and truth.shape==(206098,)
        and v012_array_sha(truth)==prior['native_scored_truth_raw_array_sha256']
        and row_hash==prior['row_ids_raw_array_sha256'])
    check('v009_A_row_ID_alignment',np.array_equal(row_ids,np.load(out9/'A_row_ids.npy',allow_pickle=False)))
    predictions={}; members={}
    for name in ['original_deep','original_compact','curve_consistency','learned_memory']:
        member=prior['members'][name]; record_path=v012_local(root,member['record_file'])
        verify(record_path,member['record_file_sha256']); record=read(record_path)
        model_path=v012_local(root,record['checkpoint_path']); verify(model_path,member['checkpoint_file_sha256'])
        path=v012_local(root,record['prediction_file']); verify(path,member['prediction_file_sha256'])
        values=np.load(path,allow_pickle=False)
        check('prior_prediction_shape_dtype_hash:'+name,values.shape==truth.shape and values.dtype==np.float64
            and np.isfinite(values).all() and v012_array_sha(values)==record['prediction_sha256']
            ==member['prediction_raw_array_sha256'] and member['row_ids_raw_array_sha256']==row_hash)
        predictions[name]=values; members[name]=member
    record_path=out9/'record_A_shared_breath_offset.json'; record=read(record_path)
    report=read(out9/'shared_breath_offset/report.json'); offset_summary=next(item for item in summary['results'] if item['name']=='shared_breath_offset')
    check('offset_record_completed_recipe',v012_file_sha(record_path)==offset_summary['record_sha256']
        and record['design_sha256']==v012_json_sha(design) and record['row_ids_sha256']==row_hash
        and record['recipe']['component_hashes']==design['component_hashes'] and record['training_breaths']==61114
        and record['validation_breaths']==6791 and record['recipe']['seed']==20261102
        and report['completed'] and report['executed_epochs']==200 and report['best_epoch']==record['best_epoch']
        and record['selected_lr_schedule']==report['selected_lr_schedule']
        ==record['recipe']['train_cfg']['frozen_lr_schedule'][:record['best_epoch']]
        and report['executed_lr_schedule']==record['recipe']['train_cfg']['frozen_lr_schedule'])
    check('offset_model_report_hashes',record['checkpoint_sha256']==report['checkpoint_sha256']
        ==v012_file_sha(out9/'shared_breath_offset/model.pt')
        and record['report_sha256']==v012_file_sha(out9/'shared_breath_offset/report.json'))
    values=np.load(out9/'A_shared_breath_offset_predictions.npy',allow_pickle=False)
    check('offset_native_prediction_shape_dtype_hash',values.shape==truth.shape and values.dtype==np.float64
        and np.isfinite(values).all() and v012_array_sha(values)==record['prediction_sha256'])
    check('offset_individual_metric_matches_completed_record',np.isclose(v012_metrics(values,truth,counts)['mae'],
        record['validation_mae'],rtol=0.,atol=1e-12))
    predictions['shared_breath_offset']=values
    members['shared_breath_offset']=dict(record_file=str(record_path.relative_to(root)),record_file_sha256=v012_file_sha(record_path),
        checkpoint_file='outputs/controlled_experiments_v009/shared_breath_offset/model.pt',checkpoint_file_sha256=record['checkpoint_sha256'],
        prediction_file='outputs/controlled_experiments_v009/A_shared_breath_offset_predictions.npy',
        prediction_file_sha256=v012_file_sha(out9/'A_shared_breath_offset_predictions.npy'),
        prediction_raw_array_sha256=record['prediction_sha256'],row_ids_raw_array_sha256=row_hash,
        best_epoch=record['best_epoch'],selected_lr_schedule=record['selected_lr_schedule'])
    reference=.5*(predictions['original_deep']+predictions['original_compact'])
    reference_metrics=v012_metrics(reference,truth,counts)
    check('unchanged_original_pair_MAE',np.isclose(reference_metrics['mae'],.19656260982217663,rtol=0.,atol=1e-12))
    check('one_exact_predeclared_roster',set(V012_WEIGHTS)==set(predictions)
        and V012_WEIGHTS['original_deep']==.5 and all(V012_WEIGHTS[k]==.125 for k in V012_WEIGHTS if k!='original_deep')
        and sum(V012_WEIGHTS.values())==1.)
    prediction=sum(weight*predictions[name] for name,weight in V012_WEIGHTS.items())
    metrics=v012_metrics(prediction,truth,counts)
    interval=v012_paired_interval(np.abs(prediction-truth),np.abs(reference-truth),counts)
    gain=reference_metrics['mae']-metrics['mae']; qualifies=bool(gain>=.003 and interval['bootstrap_95_high']<0.)
    directory.mkdir(parents=True,exist_ok=True)
    prediction_path=directory/('A_'+V012_NAME+'_predictions.npy'); rows_path=directory/'A_row_ids.npy'
    for path,array in [(prediction_path,prediction),(rows_path,row_ids)]:
        if path.exists(): raise FileExistsError('Partial V012 artifact requires review: '+str(path))
        with path.open('xb') as stream: np.save(stream,array,allow_pickle=False)
        check('saved_output_roundtrip:'+path.name,np.array_equal(np.load(path,allow_pickle=False),array))
    after=v012_code_sha(root/V012_NOTEBOOK); check('source_code_hash_unchanged',before==after)
    result=dict(name=V012_NAME,weights=V012_WEIGHTS,metrics=metrics,gain_vs_original_pair=float(gain),
        interval=interval,qualifies_provisionally=qualifies,prediction_file=str(prediction_path.relative_to(root)),
        prediction_file_sha256=v012_file_sha(prediction_path),prediction_raw_array_sha256=v012_array_sha(prediction),
        row_ids_file=str(rows_path.relative_to(root)),row_ids_file_sha256=v012_file_sha(rows_path),row_ids_raw_array_sha256=row_hash)
    report=dict(schema='fixed_diversity_screen_v012',status='complete_provisional_no_promotion',
        created_at_utc=datetime.now(timezone.utc).isoformat(),candidate_count=1,weight_or_roster_search=False,
        hypotheses_file='research/ENSEMBLE_HYPOTHESIS_v012.md',hypotheses_sha256=V012_HYPOTHESIS_SHA,
        code_hash_before_run=before,code_hash_after_run=after,notebook_code_hash_unchanged=True,
        reference=dict(weights=dict(original_deep=.5,original_compact=.5),metrics=reference_metrics),
        results=[result],members=members,verified_inputs=inputs,source_manifest=source_manifest,
        notebook_output_variations=notebook_output_variations,completed_v009_manifest_sha256=V012_COMPLETED_MANIFEST_SHA,
        prior_v010_independent_review=dict(file='research/diversity_review_v010.json',file_sha256=V012_PRIOR_REVIEW_SHA,checks=58),
        original_A_validation_breaths=6791,inspiratory_points=len(truth),row_ids_raw_array_sha256=row_hash,
        native_scored_truth_raw_array_sha256=v012_array_sha(truth),label_parse_dtype='float32',metric_arithmetic_dtype='float64',
        gate=dict(minimum_gain=.003,whole_breath_bootstrap_upper95_below_zero=True,seed=20261201,repeats=1000),
        checks_passed=len(checks),checks=checks,selected_for_confirmation=None,selection_status='deferred_pending_single_frozen_comparison_of_five_candidates',
        D_split_accessed_or_constructed=False,training_performed=False,model_loading_performed=False,inference_performed=False,
        GPU_work_performed=False,confirmation_performed=False,final_fit_performed=False,new_submission_performed=False,
        official_best_preserved=.18550,limitations=['Repeatedly used A research evidence; not independent confirmation.',
            'One new fixed roster only; no D choice or promotion is made by this notebook.'],
        runtime=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__))
    temporary=destination.with_name('results.json.tmp'); temporary.write_text(json.dumps(report,indent=2,allow_nan=False)+'\n')
    os.replace(temporary,destination)
    return report


In [2]:
def v012_review_table(report):
    return pd.DataFrame([dict(name='original_compact_deep_pair',**report['reference']['metrics'],gain=0.,upper95=None,qualifies=False)]+
        [dict(name=row['name'],**row['metrics'],gain=row['gain_vs_original_pair'],upper95=row['interval']['bootstrap_95_high'],
            qualifies=row['qualifies_provisionally']) for row in report['results']])
if V012_RUN:
    V012_REPORT=v012_run_fixed_screen()
    print(v012_review_table(V012_REPORT).to_string(index=False,float_format=lambda value:f'{value:.9f}'))
    print('Status:',V012_REPORT['status'],'Checks:',V012_REPORT['checks_passed'])
    print('Code SHA256 before/after:',V012_REPORT['code_hash_before_run'],V012_REPORT['code_hash_after_run'])
elif (V012_ROOT/V012_OUT/'results.json').exists():
    V012_REPORT=json.loads((V012_ROOT/V012_OUT/'results.json').read_text())
    print(v012_review_table(V012_REPORT).to_string(index=False))
    print('Review only; no new calculation.')
else:
    print('Review only; explicit PA3_RUN_DIVERSITY_V012=1 required for the single fixed calculation.')


                                 name         mae  median_absolute_error  p90_absolute_error  median_breath_mae        gain      upper95  qualifies
           original_compact_deep_pair 0.196562610            0.100945950         0.428331375        0.118814320 0.000000000          NaN      False
deep_half_compact_curve_memory_offset 0.190851029            0.098163784         0.422594571        0.115400883 0.005711581 -0.004389991       True
Status: complete_provisional_no_promotion Checks: 181
Code SHA256 before/after: 0b0f9e8ea399dc0e2224ea3dfc1033f7bc881fa873a1acb2de8d7fb8b66bd0e7 0b0f9e8ea399dc0e2224ea3dfc1033f7bc881fa873a1acb2de8d7fb8b66bd0e7
